<a href="https://colab.research.google.com/github/Jhessye/Adam-Sandler-ou-uma-Lesbica/blob/main/adam_lesbica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Adam sandler ou uma Lesbica ? ksks

Aqui, vamos começar subindo nossa base de testes

In [18]:
import os
import numpy as np
from PIL import Image

base_path = '/content/recortes'
classes = ['adam', 'lesbica']
amostras = {} # { ('adam', 'foto1'): { 'blusa': array_rgb, 'bermuda': array_rgb } }

for classe in classes:
    pasta_classe = os.path.join(base_path, classe)

    for arquivo in os.listdir(pasta_classe):

        partes = os.path.splitext(arquivo)[0].split('_')

        foto_id = partes[0]
        tipo_peca = partes[2]

        caminho_imagem = os.path.join(pasta_classe, arquivo)

        #rbg
        img = Image.open(caminho_imagem).convert('RGB')
        img_array = np.array(img)
        media_rgb = img_array.mean(axis=(0, 1))

        # guardando as amostras
        chave = (classe, foto_id)
        if chave not in amostras:
            amostras[chave] = {}

        amostras[chave][tipo_peca] = media_rgb

In [19]:
#armazenar as características X e rotulos y
X = []
y = []

for (classe, foto), pecas in amostras.items():
    if 'blusa' in pecas and 'bermuda' in pecas:
        vetor_blusa = pecas['blusa']
        vetor_bermuda = pecas['bermuda']

        vetor_6_valores = np.concatenate([vetor_blusa, vetor_bermuda])

        X.append(vetor_6_valores)
        y.append(classe)

        print(f"Classe: {classe:<8} | Foto: {foto:<5} | Vetor Blusa (RGB): {np.round(vetor_blusa, 1)}")
        print(f"      | Bermuda (RGB): {np.round(vetor_bermuda, 1)} -> Vetor Combinado (6 posições): {np.round(vetor_6_valores, 1)}\n")

# Convertendo as listas pra array
X = np.array(X)
y = np.array(y)

Classe: adam     | Foto: foto4 | Vetor Blusa (RGB): [179.1 158.2 187.8]
      | Bermuda (RGB): [99.7 61.8 77.4] -> Vetor Combinado (6 posições): [179.1 158.2 187.8  99.7  61.8  77.4]

Classe: adam     | Foto: foto3 | Vetor Blusa (RGB): [ 93.3 133.5 116.1]
      | Bermuda (RGB): [73.2 69.7 73. ] -> Vetor Combinado (6 posições): [ 93.3 133.5 116.1  73.2  69.7  73. ]

Classe: adam     | Foto: foto2 | Vetor Blusa (RGB): [103.9  74.2  81.6]
      | Bermuda (RGB): [ 79.2  88.2 135.4] -> Vetor Combinado (6 posições): [103.9  74.2  81.6  79.2  88.2 135.4]

Classe: adam     | Foto: foto1 | Vetor Blusa (RGB): [ 95.4  91.5 102. ]
      | Bermuda (RGB): [176.1 178.3 194.6] -> Vetor Combinado (6 posições): [ 95.4  91.5 102.  176.1 178.3 194.6]

Classe: adam     | Foto: foto5 | Vetor Blusa (RGB): [151.7 130.2 142.8]
      | Bermuda (RGB): [65.  63.  72.1] -> Vetor Combinado (6 posições): [151.7 130.2 142.8  65.   63.   72.1]

Classe: lesbica  | Foto: foto3 | Vetor Blusa (RGB): [154.4 166.6 184.5]
  

teste real agora na pasta quem.

In [20]:

pasta_quem = '/content/recortes/quem'

#nova pessoa
amostra_quem = {}

if os.path.exists(pasta_quem):
    for arquivo in os.listdir(pasta_quem):

        partes = os.path.splitext(arquivo)[0].split('_')

        foto_id = partes[0]
        tipo_peca = partes[2]

        caminho_imagem = os.path.join(pasta_quem, arquivo)

        img = Image.open(caminho_imagem).convert('RGB')
        img_array = np.array(img)
        media_rgb = img_array.mean(axis=(0, 1))

        chave = ('quem', foto_id)
        if chave not in amostra_quem:
            amostra_quem[chave] = {}
        amostra_quem[chave][tipo_peca] = media_rgb

# faze o novo vetor de 6 características
novo_x = None
nome_foto_quem = ""
for (classe, foto), pecas in amostra_quem.items():
    if 'blusa' in pecas and 'bermuda' in pecas:
        novo_x = np.concatenate([pecas['blusa'], pecas['bermuda']])
        nome_foto_quem = foto
        break

    distancias = np.linalg.norm(X - novo_x, axis=1)

    rotulos_fotos = [f"{amostra[0]} ({amostra[1]})" for amostra in amostras.keys() if 'blusa' in amostras[amostra] and 'bermuda' in amostras[amostra]]

In [21]:
from collections import Counter
import numpy as np
k = 3

indices_k = np.argsort(distancias)[:k]

classes_vizinhas = y[indices_k]
distancias_vizinhas = distancias[indices_k]

#contagem
votacao = Counter(classes_vizinhas)
classe_vencedora, votos = votacao.most_common(1)[0]

print(f"--- Resultado do KNN (K = {k}) para a foto '{nome_foto_quem}' ---\n")
print("Vizinhos mais próximos encontrados:")
for i, (idx, classe, dist) in enumerate(zip(indices_k, classes_vizinhas, distancias_vizinhas), 1):
    print(f"{i}º Vizinho: Classe: {classe:<8} | Distância: {dist:.2f}")

print(f"\n Chute Final (Predição): {classe_vencedora.upper()} ({votos} de {k} votos)")

--- Resultado do KNN (K = 3) para a foto 'adam' ---

Vizinhos mais próximos encontrados:
1º Vizinho: Classe: lesbica  | Distância: 70.02
2º Vizinho: Classe: adam     | Distância: 96.15
3º Vizinho: Classe: adam     | Distância: 107.08

 Chute Final (Predição): ADAM (2 de 3 votos)
